In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sample text data
text_data = """
Before Islamic influence started in the mid-8th century AD, Tashkent was influenced by the Sogdian and Turkic cultures. After Genghis Khan destroyed it in 1219, it was rebuilt and profited from the Silk Road. From the 18th to the 19th centuries, the city became an independent city-state, before being re-conquered by the Khanate of Kokand. In 1865, Tashkent fell to the Russian Empire; as a result, it became the capital of Russian Turkestan. In Soviet times,
it witnessed major growth and demographic changes due to forced deportations from throughout the Soviet Union. Much of Tashkent was destroyed in the 1966 Tashkent earthquake, but it was soon rebuilt as a model Soviet city.
"""

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text_data])
total_words = len(tokenizer.word_index) + 1

# Create input sequences using the tokenized text
input_sequences = []
for line in text_data.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences to have uniform length
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Create predictors and label
X, y = input_sequences[:,:-1],input_sequences[:,-1]

# Convert labels to one-hot encoding
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Model architecture
model = Sequential([
    Embedding(total_words, 100, input_length=max_sequence_len-1),
    LSTM(150, return_sequences=True),
    LSTM(150),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Model training
history = model.fit(X, y, epochs=100, verbose=1)

# Text generation function
def generate_text(seed_text, next_words, model, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted_probs = model.predict(token_list, verbose=0)[0]
        predicted = np.random.choice(range(total_words), p=predicted_probs)
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# Generate new text
seed_text = "Tashkent is"
generated_text = generate_text(seed_text, 10, model, max_sequence_len)
print(generated_text)

Epoch 1/100
4/4 [==============================] - 6s 366ms/step - loss: 4.2774 - accuracy: 0.0619
Epoch 2/100
4/4 [==============================] - 1s 253ms/step - loss: 4.2520 - accuracy: 0.0973
Epoch 3/100
4/4 [==============================] - 1s 263ms/step - loss: 4.1549 - accuracy: 0.0973
Epoch 4/100
4/4 [==============================] - 1s 258ms/step - loss: 4.0882 - accuracy: 0.0973
Epoch 5/100
4/4 [==============================] - 1s 241ms/step - loss: 4.0349 - accuracy: 0.0885
Epoch 6/100
4/4 [==============================] - 1s 248ms/step - loss: 3.9660 - accuracy: 0.0885
Epoch 7/100
4/4 [==============================] - 1s 247ms/step - loss: 3.9020 - accuracy: 0.0885
Epoch 8/100
4/4 [==============================] - 1s 247ms/step - loss: 3.8236 - accuracy: 0.0973
Epoch 9/100
4/4 [==============================] - 1s 255ms/step - loss: 3.7526 - accuracy: 0.0973
Epoch 10/100
4/4 [==============================] - 1s 241ms/step - loss: 3.7031 - accuracy: 0.0885
Epoch 11/